**Author:** Pankaj Yadav          
**Date:** 03/29/2026                 
**Description:**              
This assignment is to learn connecting GenAI API and extract the text from a PDF    

**Problem #4 - Extraction**
Using the OpenAI API, extract the text from this invoice and then use an LLM to convert the text to structured JSON (you can literally ask the LLM to produce JSON).

Build the solution using a Jupyter Notebook (no other Python is acceptable) and include Markdown that thoroughly explains your thought process and your commentary on the results you achieved. Obvious Markdown that explicitly only explains the code will be ignored, since I can read the code. I’m looking for your commentary and evaluation in your own words.

**Explanation**

First step was to install the OpenAI python library. Once installed I created a project in OpenAI's API platform and created a APi_key. Put some balance in there. I am using mac and have an separate admin account for security purpose and it is not letting me export my API key to .zshrc file hence using it as is for execution and later on I will remove it for security purpose.

I setup Json, path and openAI libraries. 
Json to convert the results from openAI , path to define the actual path for the file and OpenAI to connect and communicate with model. 

In [ ]:
import json
from pathlib import Path
from openai import OpenAI
client = OpenAI(api_key="sk-...")  # Replace with your actual API key

Once the path is defined for the pdf invoice file. client.files.create calls openai's files API to upload the defined file. I used rb while opening the file to instruct the llm to read the file as binary mode as its a pdf and raw bytes will be a better option. Purpose is just the label for the file.

Below the file read, i create a prompt variable and explained my request along with detailed rules.

Client.responses is the actual model call and combines all instructions in prompt and the PDF. Temperature = 0 is what I learnt from OpenAI's API document which makes the output less random.

In [2]:
pdf_path = Path("dsc-670-exercise-invoice.pdf")


file = client.files.create(
    file=open(pdf_path, "rb"),
    purpose="user_data"
)

prompt = """
Read this invoice PDF and return structured JSON only.

Rules:
Use null for missing values.
Do not guess unknown values.
Keep monetary values numeric where possible.
document_type must be "invoice".
"""

response = client.responses.create(
    model="gpt-4.1",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": prompt},
            {"type": "input_file", "file_id": file.id}
        ]
    }],
    temperature=0
)

result = json.loads(response.output_text)
print(json.dumps(result, indent=2))

{
  "document_type": "invoice",
  "invoice_number": "1111",
  "invoice_date": "2024-06-28",
  "purchase_order_number": "123456",
  "seller": {
    "name": null,
    "address": "321 Avenue A, Portland, OR 12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "buyer": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "address": "123 Main St., Manhattan, NY 98765",
    "phone": "(321) 555-1234",
    "email": null
  },
  "line_items": [
    {
      "quantity": 1,
      "description": "Item Number 1",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": null
    },
    {
      "quantity": 1,
      "description": "Item Number 2",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": null
    },
    {
      "quantity": 1,
      "description": "Item Number 3",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": null
    }
  ],
  "subtotal": 6.0,
  "credit":